# NOTEBOOK INVENTORY

This notebook was empty at inspection time. The inventory below reflects the assignment walkthrough in `Final Assignment/tutorial.ipynb` and a minimal reproducible validation setup for the telecom data.

## Summary

- Dataframes: `client`, `record`, `df`, `df_clean`, `X`, `y`, `X_train`, `X_test`, `y_train`, `y_test`, `model`
- Merged dataframe shape: `(100000, 100)`
- Modeling dataframe shape: `(100000, 99)`
- Target column: `churn`
- Modeling features: `98`
- Preprocessing: drop `Customer_ID`, label-encode object columns
- Feature engineering: no persistent engineered features in the modeling pipeline
- Split logic: 70/30 stratified train-test split with `random_state=42`
- Model: `XGBClassifier`


In [36]:
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

base_dir = Path("./telecom")
client = pd.read_csv(base_dir / "Client.csv")
record = pd.read_csv(base_dir / "Record.csv")

# Keep the modeling dataframe name used in the reference notebook.
df = record.merge(client, on="Customer_ID", how="inner")
df_clean = df.drop(columns=["Customer_ID"])
X = df_clean.drop(columns=["churn"])
y = df_clean["churn"]


In [37]:
print(f"Modeling dataframe name: df_clean")
print(f"Target column: churn")
print(f"Current row/column count: {df_clean.shape}")
print(f"Feature count: {X.shape[1]}")


Modeling dataframe name: df_clean
Target column: churn
Current row/column count: (100000, 99)
Feature count: 98


# MISSING VALUE TREATMENT

This section treats missing values on the existing modeling dataframe `df_clean` without reloading the data or dropping any features.

In [38]:
import pandas as pd
from pathlib import Path

# Work on the existing modeling dataframe only.
missing_value_df = df_clean.copy()
feature_cols = [col for col in missing_value_df.columns if col != 'churn']
missing_features = [col for col in feature_cols if missing_value_df[col].isna().any()]

summary_rows = []
for col in feature_cols:
    missing_before = int(missing_value_df[col].isna().sum())
    indicator_created = missing_before > 0
    imputation_strategy = 'none'

    if indicator_created:
        indicator_col = f'{col}_missing_ind'
        missing_value_df[indicator_col] = missing_value_df[col].isna().astype(int)

        if pd.api.types.is_numeric_dtype(missing_value_df[col]):
            fill_value = missing_value_df[col].median()
            missing_value_df[col] = missing_value_df[col].fillna(fill_value)
            imputation_strategy = 'median'
        else:
            missing_value_df[col] = missing_value_df[col].fillna('MISSING')
            imputation_strategy = 'MISSING'

    missing_after = int(missing_value_df[col].isna().sum())
    summary_rows.append({
        'feature': col,
        'missing_before': missing_before,
        'missing_after': missing_after,
        'indicator_created': indicator_created,
        'imputation_strategy': imputation_strategy,
    })

missingness_summary_df = pd.DataFrame(summary_rows)

# Preserve the treated dataframe as the next modeling state.
df_missingness = missing_value_df

output_dir_docs = Path('docs/4-EDA-reports')
output_dir_docs.mkdir(parents=True, exist_ok=True)
output_dir_data = Path('Final Assignment/data/intermediate')
output_dir_data.mkdir(parents=True, exist_ok=True)
output_dir_telecom = Path('Final Assignment/telecom')
output_dir_telecom.mkdir(parents=True, exist_ok=True)

snapshot_path = output_dir_data / 'E1_missingness.parquet'
summary_csv_path = output_dir_telecom / 'missingness_summary.csv'
report_path = output_dir_docs / 'missingness_features_report.md'

missingness_summary_df.to_csv(summary_csv_path, index=False)

# Pandas->pyarrow parquet can hit a notebook-specific extension-type cleanup bug.
# Writing the Arrow table directly avoids the failing unregister path.
snapshot_table = pa.Table.from_pandas(df_missingness, preserve_index=False)
pq.write_table(snapshot_table, snapshot_path)

missing_before_total = int(df_clean.isna().sum().sum())
missing_after_total = int(df_missingness.isna().sum().sum())
indicators_created = int(missingness_summary_df['indicator_created'].sum())
shape_before = df_clean.shape
shape_after = df_missingness.shape
snapshot_created = snapshot_path.exists()

report_lines = [
    '# Missing Value Treatment Report',
    '',
    '## Validation',
    '',
    f'- Missing count before treatment: {missing_before_total}',
    f'- Missing count after treatment: {missing_after_total}',
    f'- Indicators created: {indicators_created}',
    f'- Dataframe shape before: {shape_before}',
    f'- Dataframe shape after: {shape_after}',
    f'- E1_missingness.parquet created successfully: {snapshot_created}',
    '',
    '## Summary Table',
    '',
    missingness_summary_df.to_markdown(index=False),
]
report_path.write_text('\n'.join(report_lines))

print(f'Missing features treated: {len(missing_features)}')
print(f'Missing count before treatment: {missing_before_total}')
print(f'Missing count after treatment: {missing_after_total}')
print(f'Indicators created: {indicators_created}')
print(f'Shape before: {shape_before}')
print(f'Shape after: {shape_after}')
print(f'Snapshot created: {snapshot_created}')
print(f'Report: {report_path}')
print(f'Summary CSV: {summary_csv_path}')
print(f'Snapshot parquet: {snapshot_path}')


Missing features treated: 43
Missing count before treatment: 342969
Missing count after treatment: 0
Indicators created: 43
Shape before: (100000, 99)
Shape after: (100000, 142)
Snapshot created: True
Report: docs/4-EDA-reports/missingness_features_report.md
Summary CSV: Final Assignment/telecom/missingness_summary.csv
Snapshot parquet: Final Assignment/data/intermediate/E1_missingness.parquet


---

## 4.Feature Governance


In [39]:
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

# Governance classification for the current modeling dataframe.
# Safe features are stable customer/profile attributes.
# Conditional features are usable only if they are observed before the decision point.
# Unknown features are ambiguous codes or fields whose business meaning is not clear enough from EDA alone.

SAFE_FEATURES = [
    'months', 'uniqsubs', 'actvsubs', 'new_cell', 'crclscod', 'asl_flag',
    'prizm_social_one', 'area', 'dualband', 'refurb_new', 'hnd_price',
    'phones', 'models', 'hnd_webcap', 'truck', 'rv', 'ownrent', 'lor',
    'dwlltype', 'marital', 'adults', 'infobase', 'income', 'numbcars',
    'HHstatin', 'dwllsize', 'forgntvl', 'ethnic', 'kid0_2', 'kid3_5',
    'kid6_10', 'kid11_15', 'kid16_17', 'creditcd', 'eqpdays',
]

CONDITIONAL_FEATURES = [
    'rev_Mean', 'mou_Mean', 'totmrc_Mean', 'da_Mean', 'ovrmou_Mean',
    'ovrrev_Mean', 'vceovr_Mean', 'datovr_Mean', 'roam_Mean', 'change_mou',
    'change_rev', 'drop_vce_Mean', 'drop_dat_Mean', 'blck_vce_Mean',
    'blck_dat_Mean', 'unan_vce_Mean', 'unan_dat_Mean', 'plcd_vce_Mean',
    'plcd_dat_Mean', 'recv_vce_Mean', 'recv_sms_Mean', 'comp_vce_Mean',
    'comp_dat_Mean', 'custcare_Mean', 'ccrndmou_Mean', 'cc_mou_Mean',
    'inonemin_Mean', 'threeway_Mean', 'mou_cvce_Mean', 'mou_cdat_Mean',
    'mou_rvce_Mean', 'owylis_vce_Mean', 'mouowylisv_Mean', 'iwylis_vce_Mean',
    'mouiwylisv_Mean', 'peak_vce_Mean', 'peak_dat_Mean', 'mou_peav_Mean',
    'mou_pead_Mean', 'opk_vce_Mean', 'opk_dat_Mean', 'mou_opkv_Mean',
    'mou_opkd_Mean', 'drop_blk_Mean', 'attempt_Mean', 'complete_Mean',
    'callfwdv_Mean', 'callwait_Mean', 'totcalls', 'totmou', 'totrev',
    'adjrev', 'adjmou', 'adjqty', 'avgrev', 'avgmou', 'avgqty', 'avg3mou',
    'avg3qty', 'avg3rev', 'avg6mou', 'avg6qty', 'avg6rev',
]

UNKNOWN_FEATURES = []

feature_to_governance = {}
for feature in SAFE_FEATURES:
    feature_to_governance[feature] = ('safe', 'Static or customer profile attribute available before prediction.')
for feature in CONDITIONAL_FEATURES:
    feature_to_governance[feature] = ('conditional', 'Operational or usage signal that is useful only if measured before the decision point.')
for feature in UNKNOWN_FEATURES:
    feature_to_governance[feature] = ('unknown', 'Business meaning is ambiguous from EDA alone.')

rows = []
for feature in df_clean.columns:
    governance_type, reason = feature_to_governance.get(feature, ('unknown', 'Not clearly classifiable from EDA alone.'))
    rows.append({'feature': feature, 'governance_type': governance_type, 'reason': reason})

feature_governance_df = pd.DataFrame(rows)

# Preserve the current full dataframe state for traceability.
governance_snapshot_df = df_clean.copy()

safe_count = int((feature_governance_df['governance_type'] == 'safe').sum())
conditional_count = int((feature_governance_df['governance_type'] == 'conditional').sum())
unknown_count = int((feature_governance_df['governance_type'] == 'unknown').sum())

report_dir = Path('docs/4-EDA-reports')
telecom_dir = Path('Final Assignment/telecom')
intermediate_dir = Path('Final Assignment/data/intermediate')
report_dir.mkdir(parents=True, exist_ok=True)
telecom_dir.mkdir(parents=True, exist_ok=True)
intermediate_dir.mkdir(parents=True, exist_ok=True)

report_path = report_dir / 'feature_governance.md'
report_lines = [
    '# Feature Governance',
    '',
    f'- SAFE_FEATURES: {safe_count}',
    f'- CONDITIONAL_FEATURES: {conditional_count}',
    f'- UNKNOWN_FEATURES: {unknown_count}',
    '',
    '## Governance Registry',
    '',
    feature_governance_df.to_markdown(index=False),
]
report_path.write_text('\n'.join(report_lines))

csv_path = telecom_dir / 'feature_governance.csv'
feature_governance_df.to_csv(csv_path, index=False)
parquet_path = intermediate_dir / 'E2_governance.parquet'
# Pandas->pyarrow parquet can fail when Arrow tries to inspect notebook-defined
# Python objects or dynamically executed source. Writing the Arrow table directly
# avoids that source-introspection path.
governance_snapshot_table = pa.Table.from_pandas(governance_snapshot_df, preserve_index=False)
pq.write_table(governance_snapshot_table, parquet_path)

print('SAFE_FEATURES:', safe_count)
print('CONDITIONAL_FEATURES:', conditional_count)
print('UNKNOWN_FEATURES:', unknown_count)
print(f'feature_governance.csv created: {csv_path.exists()}')
print(f'E2_governance.parquet created: {parquet_path.exists()}')
print(f'Report: {report_path}')


SAFE_FEATURES: 35
CONDITIONAL_FEATURES: 63
UNKNOWN_FEATURES: 1
feature_governance.csv created: True
E2_governance.parquet created: True
Report: docs/4-EDA-reports/feature_governance.md


In [40]:
print(feature_governance_df.head(10).to_string(index=False))
print('Total features:', len(feature_governance_df))
print('Governance counts:', feature_governance_df['governance_type'].value_counts().to_dict())


    feature governance_type                                                                                 reason
   rev_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
   mou_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
totmrc_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
    da_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
ovrmou_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
ovrrev_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
vceovr_Mean     conditional Operational or usage signal that is useful only if measured before the decision point.
datovr_Mean     conditional Operational or usage signal that is useful only if m